In [ ]:
# import sys
# sys.executable

'/Users/linlin/Documents/investment/algoTrading/venv/bin/python'

In [ ]:
# !pip list | grep numpy
# !pip list | grep pandas-ta
# !pip install numpy pandas-ta
# !pip install alpha_vantage
# !pip install xgboost

In [ ]:
# python -c "import sys; print(sys.version)"

In [1]:
import ta
# import yfinance as yf
from alpha_vantage.timeseries import TimeSeries
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error
# import xgboost as xgb
from backtesting import Backtest#, Strategy
from backtesting.lib import Strategy
import strategies
import ta.trend
import ta.momentum

/Users/linlin/Documents/investment/algoTrading/venv/lib/python3.10/site-packages/backtesting/_plotting.py:53: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [11]:
# import time
from datetime import datetime, timedelta

Parallelization Method:
|Method|Best For|Complexity|
|:-----|:-------|:----------|
|Scikit-Learn (`n_jobs=-1`)| Multi-threading, fastest for local machine| Simple|
|Joblib(`Parallel`)| Multiple stock models in parallel| Simple|
|Dask (`GridSearchCV`)|Large datasets, multi-node computing| Intermediate|
|Spark MLlib (`PySpark`)|Massive datasets, distributed computing| Complex|

In [2]:
from joblib import Parallel, delayed

In [61]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

## 1. Fetch data
Source: [Alpha Vantage](https://www.alphavantage.co/documentation/)

Watchlist: 
{"Retail": ['COST', 'KR', 'KO', 'HD'], <br>
"Finance": ['JPM', 'BRK-B', 'V', 'AXP', 'COIN'],<br>
"Semiconductor": ['NVDA', 'AVGO', 'AMD', 'ASML'], <br>
"Auto":['TM', 'HMC', RACE', 'VOW3.DE'], <br>
"Software":['TOST', 'UBER', 'NET', 'XYZ', 'ORCL', 'SAP', 'PLTR', 'ADBE', 'CRWD'], <br>
"Energy":['NEE', 'CVX', 'PLUG']} <br>
'PSTG' <br>
['AAPL', 'MSFT', 'AMZN', 'META', 'GOOGL']

In [2]:
API_KEY= "JQ88029HVHREY29X"

In [3]:
# Initiate Alpha Vantage API
ts = TimeSeries(key=API_KEY, output_format='pandas')

In [13]:
stocks = ['AAPL', 'MSFT', 'AMZN', 'META', 'GOOGL']

dt = {}

# Define past 10 years data
ten_yr = datetime.today() - timedelta(days=10 * 365)

for stock in stocks:
    try:
        df, meta_data = ts.get_daily(symbol=stock, outputsize='full')
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        df = df[::-1] #reverse order
        
        # Conver index to datetime & filter date
        df.index = pd.to_datetime(df.index)
        df = df[df.index >= ten_yr]
        
        dt[stock] = df
    except Exception as e:
        print(f"Error downloading {stock}: {e}")

# method1: set time out (failed)
# dt = {s: yf.download(s, start='2016-01-01', end='2025-03-05', timeout=20) for s in stocks}
# dt

# method2: add sleep time and disable multithreading (failed)
# dt = {}

# for stock in stocks:
#     try:
#         dt[stock] = yf.download(stock, start='2016-01-01', end='2025-03-05', threads=False) #to disable multi-threading
#         time.sleep(1) # pause for 1 seconds between downloads as YFinance prevents too many fast requests
#     except Exception as e:
#         print(f"Error downloading {stock}: {e}")


In [19]:
dt

{'AAPL':                Open      High      Low   Close      Volume
 date                                                      
 2015-03-11  124.750  124.7700  122.110  122.24  68938974.0
 2015-03-12  122.310  124.9000  121.630  124.45  48362719.0
 2015-03-13  124.400  125.3951  122.580  123.59  51827283.0
 2015-03-16  123.880  124.9500  122.870  124.95  35874300.0
 2015-03-17  125.900  127.3200  125.650  127.04  51023104.0
 ...             ...       ...      ...     ...         ...
 2025-02-27  239.410  242.4600  237.060  237.30  41153639.0
 2025-02-28  236.950  242.0900  230.200  241.84  56833360.0
 2025-03-03  241.790  244.0272  236.112  238.03  47183985.0
 2025-03-04  237.705  240.0700  234.680  235.93  53798062.0
 2025-03-05  235.420  236.5500  229.230  235.74  47227643.0
 
 [2512 rows x 5 columns],
 'MSFT':                Open    High      Low    Close      Volume
 date                                                     
 2015-03-11   42.310   42.37   41.840   41.980  32215314.0

Convert dictionary for processing

In [14]:
df_big5 = pd.concat(dt, names=['Stock', 'Date'])
df_big5.head()

Open      High     Low   Close      Volume
Stock Date                                                    
AAPL  2015-03-11  124.75  124.7700  122.11  122.24  68938974.0
      2015-03-12  122.31  124.9000  121.63  124.45  48362719.0
      2015-03-13  124.40  125.3951  122.58  123.59  51827283.0
      2015-03-16  123.88  124.9500  122.87  124.95  35874300.0
      2015-03-17  125.90  127.3200  125.65  127.04  51023104.0

In [15]:
df_big5.tail()

Open      High     Low   Close      Volume
Stock Date                                                     
GOOGL 2025-02-27  173.990  174.5600  167.94  168.50  39991015.0
      2025-02-28  168.680  170.6100  166.77  170.28  48130565.0
      2025-03-03  171.925  173.3700  165.93  167.01  40770451.0
      2025-03-04  166.240  173.2946  165.80  170.92  45387996.0
      2025-03-05  170.520  173.7800  169.06  173.02  30954922.0

In [16]:
df_big5.shape

(12560, 5)

In [17]:
df_big5.to_csv('data/df_big5.csv')

## 2. Feature Engineering 
Technical Indicators:
* SMA: 50-day, 200-day
* EMA: react faster to price changes compared to SMA
* RSI= Relative Strength Index
* MACD= Moving Average Convergence Divergence
* Volatility: Rolling Std Deviation

In [18]:
def add_indicators(df):
    # Compute indicators
    df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50) #50-Day simple moving average
    df['SMA_200'] = ta.trend.sma_indicator(df['Close'], window=200)
    df['EMA_50'] = ta.trend.ema_indicator(df['Close'], window=50) #50-Day exponentional moving average
    df['EMA_200'] = ta.trend.ema_indicator(df['Close'], window=200)
    df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()#RSI= Relative Strength Index
    df['MACD'] = ta.trend.MACD(df['Close']).macd()#MACD= Moving Average Convergence Divergence
    df['Volatility'] = df['Close'].pct_change().rolling(21).std()
    
    df = df.dropna()
    
    return df
    

In [25]:
dt.items()

dict_items([('AAPL',                Open      High      Low   Close      Volume
date                                                      
2015-03-11  124.750  124.7700  122.110  122.24  68938974.0
2015-03-12  122.310  124.9000  121.630  124.45  48362719.0
2015-03-13  124.400  125.3951  122.580  123.59  51827283.0
2015-03-16  123.880  124.9500  122.870  124.95  35874300.0
2015-03-17  125.900  127.3200  125.650  127.04  51023104.0
...             ...       ...      ...     ...         ...
2025-02-27  239.410  242.4600  237.060  237.30  41153639.0
2025-02-28  236.950  242.0900  230.200  241.84  56833360.0
2025-03-03  241.790  244.0272  236.112  238.03  47183985.0
2025-03-04  237.705  240.0700  234.680  235.93  53798062.0
2025-03-05  235.420  236.5500  229.230  235.74  47227643.0

[2512 rows x 5 columns]), ('MSFT',                Open    High      Low    Close      Volume
date                                                     
2015-03-11   42.310   42.37   41.840   41.980  32215314.0
20

In [27]:
# Apply indicators to all stocks
dt_ind = {stock: add_indicators(df_big5) for stock, df in dt.items()}

In [30]:
dt_ind

{'AAPL':                      Open      High       Low   Close      Volume    SMA_50  \
 Stock Date                                                                    
 AAPL  2015-12-22  107.400  107.7200  106.4510  107.23  32789367.0  115.6865   
       2015-12-23  107.270  108.8500  107.2000  108.61  32657354.0  115.6229   
       2015-12-24  109.000  109.0000  107.9500  108.03  13596680.0  115.5793   
       2015-12-28  107.590  107.6900  106.1807  106.82  26704210.0  115.4785   
       2015-12-29  106.960  109.4300  106.8600  108.74  30931243.0  115.4325   
 ...                   ...       ...       ...     ...         ...       ...   
 GOOGL 2025-02-27  173.990  174.5600  167.9400  168.50  39991015.0  190.9678   
       2025-02-28  168.680  170.6100  166.7700  170.28  48130565.0  190.5770   
       2025-03-03  171.925  173.3700  165.9300  167.01  40770451.0  189.9840   
       2025-03-04  166.240  173.2946  165.8000  170.92  45387996.0  189.4940   
       2025-03-05  170.520  173.

In [31]:
dt_ind.keys()

dict_keys(['AAPL', 'MSFT', 'AMZN', 'META', 'GOOGL'])

In [43]:
type(dt_ind['AAPL'])

pandas.core.frame.DataFrame

In [46]:
df_big5_ind = pd.concat(dt_ind, keys=dt_ind.keys(), names=['Stock'])
df_big5_ind.reset_index(level=0, inplace=True) #remove extra stock index
df_big5_ind.head()

Stock    Open    High       Low   Close      Volume  \
Stock Date                                                             
AAPL  2015-12-22  AAPL  107.40  107.72  106.4510  107.23  32789367.0   
      2015-12-23  AAPL  107.27  108.85  107.2000  108.61  32657354.0   
      2015-12-24  AAPL  109.00  109.00  107.9500  108.03  13596680.0   
      2015-12-28  AAPL  107.59  107.69  106.1807  106.82  26704210.0   
      2015-12-29  AAPL  106.96  109.43  106.8600  108.74  30931243.0   

                    SMA_50    SMA_200      EMA_50     EMA_200        RSI  \
Stock Date                                                                 
AAPL  2015-12-22  115.6865  120.58620  114.641932  118.368852  33.205554   
      2015-12-23  115.6229  120.51805  114.405385  118.271749  37.763094   
      2015-12-24  115.5793  120.43595  114.155370  118.169841  36.631784   
      2015-12-28  115.4785  120.35210  113.867709  118.056907  34.321713   
      2015-12-29  115.4325  120.27105  113.666622  117.964202  40.710860   

                      MACD  Volatility  
Stock Date                              
AAPL  2015-12-22 -2.464286    0.014334  
      2015-12-23 -2.472782    0.014716  
      2015-12-24 -2.497526    0.014397  
      2015-12-28 -2.584975    0.014462  
      2015-12-29 -2.470868    0.015277

In [48]:
df_big5_ind.to_csv('data/df_big5_ind.csv')

In [50]:
df_big5_ind.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 61805 entries, ('AAPL', Timestamp('2015-12-22 00:00:00')) to ('GOOGL', Timestamp('2025-03-05 00:00:00'))
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Stock       61805 non-null  object 
 1   Open        61805 non-null  float64
 2   High        61805 non-null  float64
 3   Low         61805 non-null  float64
 4   Close       61805 non-null  float64
 5   Volume      61805 non-null  float64
 6   SMA_50      61805 non-null  float64
 7   SMA_200     61805 non-null  float64
 8   EMA_50      61805 non-null  float64
 9   EMA_200     61805 non-null  float64
 10  RSI         61805 non-null  float64
 11  MACD        61805 non-null  float64
 12  Volatility  61805 non-null  float64
dtypes: float64(12), object(1)
memory usage: 6.4+ MB


In [51]:
df_big5_ind.columns

Index(['Stock', 'Open', 'High', 'Low', 'Close', 'Volume', 'SMA_50', 'SMA_200',
       'EMA_50', 'EMA_200', 'RSI', 'MACD', 'Volatility'],
      dtype='object')

In [53]:
df = df_big5_ind.copy()

In [5]:
df = pd.read_csv('data/df_big5_ind.csv')
df.head()

,Stock,Date,Stock.1,Open,High,Low,Close,Volume,SMA_50,SMA_200,EMA_50,EMA_200,RSI,MACD,Volatility
0,AAPL,2015-12-22,AAPL,107.40,107.72,106.4510,107.23,32789367.0,115.6865,120.58620,114.641932,118.368852,33.205554,-2.464286,0.014334
1,AAPL,2015-12-23,AAPL,107.27,108.85,107.2000,108.61,32657354.0,115.6229,120.51805,114.405385,118.271749,37.763094,-2.472782,0.014716
2,AAPL,2015-12-24,AAPL,109.00,109.00,107.9500,108.03,13596680.0,115.5793,120.43595,114.155370,118.169841,36.631784,-2.497526,0.014397
3,AAPL,2015-12-28,AAPL,107.59,107.69,106.1807,106.82,26704210.0,115.4785,120.35210,113.867709,118.056907,34.321713,-2.584975,0.014462
4,AAPL,2015-12-29,AAPL,106.96,109.43,106.8600,108.74,30931243.0,115.4325,120.27105,113.666622,117.964202,40.710860,-2.470868,0.015277


## 2. Train ML Models
Random Forest

In [6]:
features = ['SMA_50', 'SMA_200','EMA_50', 'EMA_200', 'RSI', 'MACD', 'Volatility']
target = 'Close'

In [7]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20]
}

Method 1. Multiprocessing with `Joblib`

In [17]:
def train_and_search(stock):
    stock_df = df[df['Stock'] == stock].copy()
    stock_df['Target'] = stock_df[target].shift(-1)#predict next day's price
    stock_df.dropna(inplace=True)
    
    X_train, X_test, y_train, y_test = train_test_split(stock_df[features], stock_df['Target'], test_size=0.3, shuffle=False)
    
    # Grid search Random Forest
    grid_search = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid, 
        cv=3,
        scoring='neg_mean_squared_error', #'neg_root_mean_squared_error', #rmse
        n_jobs=-1,#run all available CPU cores
        verbose=2
    )
    grid_search.fit(X_train, y_train)
    
    # Get the best model from grid search
    best_model = grid_search.best_estimator_
    
    # Make predictions on the test set
    y_pred = best_model.predict(X_test)
    
    best_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # best_rmse = -grid_search.best_score_ #convert negative RMSE back to positive
    
    return stock, best_model, best_rmse

In [ ]:
# train_and_search('AAPL')

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END .......................max_depth=5, n_estimators=50; total time=   1.4s
[CV] END .......................max_depth=5, n_estimators=50; total time=   1.5s
[CV] END .......................max_depth=5, n_estimators=50; total time=   1.5s
[CV] END ......................max_depth=5, n_estimators=100; total time=   2.6s
[CV] END ......................max_depth=5, n_estimators=100; total time=   2.7s
[CV] END ......................max_depth=5, n_estimators=100; total time=   2.7s
[CV] END ......................max_depth=10, n_estimators=50; total time=   1.9s
[CV] END ......................max_depth=10, n_estimators=50; total time=   2.0s
[CV] END ......................max_depth=10, n_estimators=50; total time=   2.1s
[CV] END ......................max_depth=5, n_estimators=200; total time=   5.1s
[CV] END ......................max_depth=5, n_estimators=200; total time=   5.2s
[CV] END ......................max_depth=5, n_est

('AAPL',
 RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42))

In [18]:
# Run parallel execution
results = Parallel(n_jobs=-1)(delayed(train_and_search)(stock) for stock in df['Stock'].unique())

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.5s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.8s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.9s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.9s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.9s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.9s
[CV] END .......................max_depth=5, n_estimators=50; total time=   5.9s
[CV] END .......................max_depth=5, n_estimators=50; total time=   6.1s
[CV] END .......................max_depth=5, n_estim

In [19]:
results

[('AAPL',
  RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42),
  np.float64(0.6853119962557134)),
 ('MSFT',
  RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42),
  np.float64(0.6341412197305439)),
 ('AMZN',
  RandomForestRegressor(max_depth=20, random_state=42),
  np.float64(3.9935403689975195)),
 ('META',
  RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42),
  np.float64(0.828116327960651)),
 ('GOOGL',
  RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42),
  np.float64(3.1597241783048102))]

* `Parallel(n_jobs=-1)` runs multiple jobs in parallel using all available CPU cores
* `(delayed(train_and_search)(stock) for stock in df['Stock'].unique())` creates a list of function calls (`train_and_search)(stock)`) without executing them immediately
* `delayed()` function wraps `train_and_search(stock)` so `Parallel()` could execute them
* The output should be a list of tuples.

In [20]:
print(type(results))
print(len(results))
print(results[:3])

<class 'list'>
5
[('AAPL', RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42), np.float64(0.6853119962557134)), ('MSFT', RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42), np.float64(0.6341412197305439)), ('AMZN', RandomForestRegressor(max_depth=20, random_state=42), np.float64(3.9935403689975195))]


In [21]:
# Convert to dictionary for easy access
best_models = {stock: model for stock, model, rmse in results}
best_model_rmse = {stock: rmse for stock, model, rmse in results}

sorted(best_model_rmse.items(), key=lambda x: x[1])

[('MSFT', np.float64(0.6341412197305439)),
 ('AAPL', np.float64(0.6853119962557134)),
 ('META', np.float64(0.828116327960651)),
 ('GOOGL', np.float64(3.1597241783048102)),
 ('AMZN', np.float64(3.9935403689975195))]

Method 2. Spark MLlib `pyspark`

In [ ]:
# Initialize Spark
spark = SparkSession.builder.appName('StockML').getOrCreate()
# Convert DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

In [ ]:
# Define model
rf = RandomForestRegressor(featuresCol='features', labelCol='Target')

# Define parameter grid
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 200]) \
    .addGrid(rf.maxDepth, [5, 10, 20]) \
    .build()    

# Define cross validation
cv = CrossValidator(estimator=rf, estimatorParamMaps=param_grid, evaluator=RegressionEvaluator(labelCol='Target'), numFolds=3)

# Train model
cv_model = cv.fit(spark_df)

spark.stop()

If not using Spark:

In [ ]:
best_models = {}

for stock in df['Stock'].unique():
    stock_df = df[df['Stock'] == stock].copy()
    stock_df['Target'] = stock_df[target].shift(-1)#predict next day's price
    stock_df.dropna(inplace=True)
    
    X_train, X_test, y_train, y_test = train_test_split(stock_df[features], stock_df['Target'], test_size=0.3, shuffle=False)
    
    # Grid search Random Forest
    grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3)
    grid_search.fit(X_train, y_train)
    
    # Train XGBoost
    # xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=10, random_state=42)
    # xgb_model.fit(X_train, y_train)
    
    best_models[stock] = grid_search.best_estimator_

In [60]:
# best param found
best_params = grid_search.best_params_
print("Best parameters", best_params)
print("Best ROC-AUC", grid_search.best_score_)

Best parameters {'max_depth': 20, 'n_estimators': 50}
Best ROC-AUC 0.6623588612596517


## 3. Apply walk-forward validation

In [ ]:
# Define walk-forward validation strategy
tscv = TimeSeriesSplit(n_splits=5)

walk_forward_results = {}

for stock in df['Stock'].unique():
    stock_df = df[df['Stock'] == stock].copy()
    stock_df['Target'] = stock_df[target].shift(-1)#predict next day's price
    stock_df.dropna(inplace=True)
    
    X = stock_df[features]
    y = stock_df['Target']
    
    errors = []
    
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model = RandomForestRegressor(objective='reg:squarederror', n_estimators=50, max_depth=20, random_state=42, cv=3)
        model.fit(X_train, y_train)
        
        predictions = model.predict(X_test)
        error = mean_squared_error(y_test, predictions)
        errors.append(error)
    
    walk_forward_results[stock] = sum(errors) / len(errors)
    
pd.DataFrame(walk_forward_results, index=['MSE']).T

## 4. Backtest in Bull, Bear and Sideways Markets
### Define Market Scenarios

In [ ]:
def classify_market(df):
    df['Return'] = df['Close'].pct_change()
    df['Market_trend'] = 'Sideways'
    
    bull_threashold = 0.02 # Market uptrend if return > 2%
    bear_threashold = -0.02 # Market downtred if return < -2%
    
    df.loc[df['Return'] > bull_threashold, "Market_trend"] = "Bull"
    df.loc[df['Return'] < bull_threashold, "Market_trend"] = "Bear"

    return df

# Apply market classification
df_big5_ind = classify_market(df_big5_ind)

### Backtest Trading Strategy

## . Expand Strategy Parameters & Metrics

## 4. 